# 03 - Lemmatization, Stemming, Stop Words & When *Not* To

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain **stemming** as rule-based suffix stripping and implement a simple stemmer.
- Explain **lemmatization** as dictionary-based reduction to valid base forms.
- Compare stemming vs lemmatization and choose the right one for your task.
- Understand **stop word removal**: when it helps (BoW/TF-IDF) and when it **hurts** (deep learning).
- Build preprocessing pipelines that make informed choices about these techniques.

## Prerequisites

- Completed Notebook 01 (text preprocessing) and 02 (BoW/TF-IDF).
- Basic Python string operations and regex.
- Familiarity with sklearn vectorizers.

## Table of Contents

1. [Stemming: Rule-Based Suffix Stripping](#1-stemming-rule-based-suffix-stripping)
2. [Lemmatization: Dictionary-Based Reduction](#2-lemmatization-dictionary-based-reduction)
3. [Stemming vs Lemmatization: When to Use Which](#3-stemming-vs-lemmatization-when-to-use-which)
4. [Stop Words: Removal and Its Consequences](#4-stop-words-removal-and-its-consequences)
5. [Code: Implement Simple Stemmer Rules](#5-code-implement-simple-stemmer-rules)
6. [Code: Optional NLTK Lemmatization](#6-code-optional-nltk-lemmatization)
7. [Code: Stop Word Removal Effect on Classification](#7-code-stop-word-removal-effect-on-classification)
8. [Common Mistakes & Debugging Tips](#8-common-mistakes--debugging-tips)
9. [Exercises](#9-exercises)
10. [Exercise Solutions](#10-exercise-solutions)

In [ ]:
# ---- Environment setup ----
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

set_global_seed(42)

import re
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple

print("Setup complete.")

## 1. Stemming: Rule-Based Suffix Stripping

**Stemming** reduces words to their *stem* by stripping suffixes using hand-crafted rules. The result is **not necessarily a valid dictionary word**.

### How It Works

A stemmer applies a cascade of suffix-removal rules:

| Input | Suffix Rule | Stem |
|-------|-------------|------|
| running | -ning -> -n (if preceded by consonant+vowel) | run |
| cats | -s -> (empty) | cat |
| university | -ity -> (empty) | univers |
| happiness | -ness -> (empty) | happi |
| studies | -ies -> -i | studi |

### Popular Stemmers

- **Porter Stemmer** (1980): The classic. ~60 rules in 5 steps. Aggressive.
- **Snowball (Porter2)**: Improved version with better handling of edge cases.
- **Lancaster**: Even more aggressive than Porter.

### Key Properties

- **Fast**: Just string manipulation, no dictionary lookup.
- **Lossy**: `"university"` -> `"univers"` (not a real word).
- **Over-stemming**: Different words collapse to the same stem (`"universal"`, `"university"` -> `"univers"`).
- **Under-stemming**: Related words keep different stems (`"alumnus"`, `"alumni"` may not match).

In [ ]:
# ---- Demonstrate stemming behavior ----

# Examples of what a Porter-style stemmer produces
porter_examples = {
    "running": "run",
    "runs": "run",
    "ran": "ran",           # irregular - stemmer can't always handle this
    "cats": "cat",
    "university": "univers", # NOT a valid word!
    "universal": "univers",  # same stem as university
    "happiness": "happi",    # NOT a valid word!
    "happily": "happili",    # NOT a valid word!
    "studies": "studi",
    "studying": "studi",
    "better": "better",      # irregular comparative - stemmer misses it
    "connection": "connect",
    "connections": "connect",
    "connected": "connect",
}

print("Porter Stemmer Examples:")
print(f"{'Input':<15} {'Stem':<15} {'Valid word?'}")
print("-" * 45)

# A quick dictionary of common English words for checking validity
common_words = {
    "run", "ran", "cat", "university", "universal", "happiness", "happy",
    "happily", "study", "studies", "studying", "better", "good", "connect",
    "connection", "connected", "univers", "happi", "happili", "studi",
}
# Only these are actual words
real_words = {
    "run", "ran", "cat", "university", "universal", "happiness", "happy",
    "happily", "study", "studies", "studying", "better", "good", "connect",
    "connection", "connected",
}

for word, stem in porter_examples.items():
    valid = "Yes" if stem in real_words else "No"
    print(f"{word:<15} {stem:<15} {valid}")

## 2. Lemmatization: Dictionary-Based Reduction

**Lemmatization** reduces words to their *lemma* (dictionary base form) using a vocabulary lookup and morphological analysis. The result is **always a valid word**.

### How It Works

A lemmatizer uses a dictionary or morphological database:

| Input | POS | Lemma | Method |
|-------|-----|-------|--------|
| running | verb | run | Dictionary lookup |
| better | adj | good | Irregular form table |
| mice | noun | mouse | Irregular plural table |
| studies | noun | study | Suffix rule + validation |
| was | verb | be | Irregular verb table |
| universities | noun | university | Suffix rule + validation |

### Key Properties

- **Accurate**: Always returns a valid dictionary word.
- **Slower**: Requires dictionary/database lookup.
- **POS-aware**: `"better"` as adjective -> `"good"`, as adverb -> `"well"`.
- **Handles irregulars**: `"ran"` -> `"run"`, `"mice"` -> `"mouse"`.

### Common Lemmatizers

- **WordNet Lemmatizer** (NLTK): Uses WordNet database.
- **spaCy Lemmatizer**: Fast, uses lookup tables and rules.
- **Stanza**: Stanford NLP, high accuracy.

In [ ]:
# ---- Demonstrate lemmatization behavior ----

lemma_examples = {
    "running": "run",
    "runs": "run",
    "ran": "run",            # handles irregular verbs!
    "cats": "cat",
    "university": "university",  # already a base form
    "universities": "university",
    "happiness": "happiness",    # base noun form
    "happily": "happily",        # base adverb form
    "studies": "study",
    "studying": "study",
    "better": "good",         # irregular comparative!
    "mice": "mouse",          # irregular plural!
    "was": "be",              # irregular verb!
    "geese": "goose",         # irregular plural!
}

print("Lemmatization Examples:")
print(f"{'Input':<15} {'Lemma':<15} {'Valid word?'}")
print("-" * 45)
for word, lemma in lemma_examples.items():
    print(f"{word:<15} {lemma:<15} Always Yes")

## 3. Stemming vs Lemmatization: When to Use Which

| Aspect | **Stemming** | **Lemmatization** |
|--------|:------------:|:-----------------:|
| **Method** | Rule-based suffix stripping | Dictionary + morphological analysis |
| **Output** | May not be a real word | Always a valid word |
| **Speed** | Very fast | Slower (dictionary lookup) |
| **Accuracy** | Lower (over/under-stemming) | Higher |
| **Irregular forms** | Often fails (`ran` -> `ran`) | Handles well (`ran` -> `run`) |
| **POS-aware** | No | Yes (if POS tags provided) |
| **Use when** | Speed matters, BoW/TF-IDF, IR | Accuracy matters, downstream NLP |

### Decision Guide

- **Use stemming** for:
  - Information retrieval / search engines
  - Bag-of-Words / TF-IDF pipelines (where exact word form matters less)
  - Large-scale processing where speed is critical
  - Quick prototyping

- **Use lemmatization** for:
  - Text classification where meaning matters
  - Question answering, chatbots
  - When you need interpretable features
  - Sentiment analysis

- **Use neither** for:
  - Transformer models (BERT, GPT) -- they use subword tokenization
  - Named Entity Recognition
  - When preserving morphological information is important

In [ ]:
# ---- Side-by-side comparison ----

comparison_words = [
    "running", "university", "better", "mice", "studies",
    "happiness", "connected", "was", "geese", "caring",
]

# Simulated outputs (since we build our own stemmer below)
stem_results = {
    "running": "run", "university": "univers", "better": "better",
    "mice": "mice", "studies": "studi", "happiness": "happi",
    "connected": "connect", "was": "wa", "geese": "gees", "caring": "care",
}
lemma_results = {
    "running": "run", "university": "university", "better": "good",
    "mice": "mouse", "studies": "study", "happiness": "happiness",
    "connected": "connect", "was": "be", "geese": "goose", "caring": "care",
}

print(f"{'Word':<15} {'Stemmed':<15} {'Lemmatized':<15} {'Same?'}")
print("-" * 55)
for word in comparison_words:
    s = stem_results[word]
    l = lemma_results[word]
    same = "Yes" if s == l else "No"
    print(f"{word:<15} {s:<15} {l:<15} {same}")

## 4. Stop Words: Removal and Its Consequences

**Stop words** are high-frequency, low-information words: *the, is, at, which, on, a, an, and, ...*

### When Stop Word Removal **Helps**

- **BoW / TF-IDF**: Stop words dominate counts and add noise.
- **Topic modeling (LDA)**: Stop words obscure meaningful topics.
- **Information retrieval**: Reduces index size, speeds up search.

### When Stop Word Removal **Hurts**

- **Deep learning / transformers**: Context words like `"not"`, `"no"`, `"but"` carry critical meaning.
  - `"not good"` -> remove `"not"` -> `"good"` (meaning reversed!)
  - `"no problem"` -> remove `"no"` -> `"problem"` (meaning reversed!)
- **Phrase-based tasks**: `"to be or not to be"` -> `"be be"` (meaningless).
- **Named entities**: `"The Who"` (band), `"The Hague"` (city).
- **Question answering**: `"who"`, `"what"`, `"where"` are stop words but critical for QA.

> **Rule of thumb:** Remove stop words for BoW/TF-IDF. **Keep** them for deep learning models.

In [ ]:
# ---- Define a standard English stop word list ----

STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "need", "dare", "ought",
    "used", "to", "of", "in", "for", "on", "with", "at", "by", "from",
    "as", "into", "through", "during", "before", "after", "above",
    "below", "between", "out", "off", "over", "under", "again",
    "further", "then", "once", "and", "but", "or", "nor", "not", "no",
    "so", "than", "too", "very", "just", "about", "up", "down",
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs",
    "themselves", "what", "which", "who", "whom", "this", "that",
    "these", "those", "am", "if", "because", "until", "while",
    "here", "there", "when", "where", "why", "how", "all", "each",
    "every", "both", "few", "more", "most", "other", "some", "such",
    "only", "own", "same", "also",
}

def remove_stop_words(tokens: List[str], stop_words: set = STOP_WORDS) -> List[str]:
    """Remove stop words from a list of tokens."""
    return [t for t in tokens if t.lower() not in stop_words]

# Demonstrate the problem
examples = [
    "This movie is not good at all",
    "No problem with the service",
    "I will not be going to the party",
    "To be or not to be that is the question",
]

print("Stop Word Removal -- When It Goes Wrong:")
print("=" * 60)
for text in examples:
    tokens = text.lower().split()
    filtered = remove_stop_words(tokens)
    print(f"  Original:  {text}")
    print(f"  Filtered:  {' '.join(filtered)}")
    print()

## 5. Code: Implement Simple Stemmer Rules

We build a simplified Porter-style stemmer from scratch to understand the mechanics.

In [ ]:
class SimpleStemmer:
    """A simplified Porter-style stemmer for educational purposes.
    
    Implements a subset of suffix-stripping rules to demonstrate
    how stemming works. NOT production-quality.
    """
    
    def __init__(self):
        # Rules: (suffix, replacement, min_stem_length)
        # Applied in order; first match wins per step
        self.step1_rules = [
            ("sses", "ss", 0),    # caresses -> caress
            ("ies", "i", 0),       # ponies -> poni
            ("ss", "ss", 0),       # caress -> caress (no change)
            ("s", "", 1),           # cats -> cat
        ]
        self.step2_rules = [
            ("eed", "ee", 1),      # agreed -> agree
            ("ed", "", 2),          # plastered -> plaster
            ("ing", "", 2),         # motoring -> motor
        ]
        self.step3_rules = [
            ("ational", "ate", 0),  # relational -> relate
            ("tional", "tion", 0),  # conditional -> condition
            ("izer", "ize", 0),     # digitizer -> digitize
            ("iveness", "ive", 0),  # effectiveness -> effective
            ("fulness", "ful", 0),  # hopefulness -> hopeful
            ("ousness", "ous", 0),  # callousness -> callous
            ("ness", "", 2),        # happiness -> happi (after step1: happi)
            ("ment", "", 2),        # adjustment -> adjust
            ("ity", "", 2),         # university -> univers
            ("ation", "ate", 0),    # activation -> activate
            ("ator", "ate", 0),     # operator -> operate
            ("ly", "", 2),          # happily -> happi (after other steps)
        ]
    
    def _apply_rules(self, word: str, rules: list) -> str:
        """Apply first matching rule from the rule list."""
        for suffix, replacement, min_len in rules:
            if word.endswith(suffix):
                stem = word[:-len(suffix)]
                if len(stem) >= min_len:
                    return stem + replacement
        return word
    
    def stem(self, word: str) -> str:
        """Stem a single word."""
        word = word.lower().strip()
        if len(word) <= 2:
            return word
        
        # Apply rules in steps
        word = self._apply_rules(word, self.step1_rules)
        word = self._apply_rules(word, self.step2_rules)
        word = self._apply_rules(word, self.step3_rules)
        
        return word
    
    def stem_tokens(self, tokens: List[str]) -> List[str]:
        """Stem a list of tokens."""
        return [self.stem(t) for t in tokens]


# Test our stemmer
stemmer = SimpleStemmer()

test_words = [
    "running", "runs", "cats", "caresses", "ponies",
    "university", "happiness", "connected", "adjustment",
    "relational", "effectiveness", "digitizer", "motoring",
    "agreed", "plastered", "hopefulness", "activation",
]

print("SimpleStemmer Results:")
print(f"{'Input':<20} {'Stem':<20}")
print("-" * 40)
for word in test_words:
    print(f"{word:<20} {stemmer.stem(word):<20}")

In [ ]:
# ---- Show stemming applied to a sentence ----

sentence = "The universities are studying the effectiveness of connected learning systems"
tokens = sentence.lower().split()
stemmed = stemmer.stem_tokens(tokens)

print(f"Original: {sentence}")
print(f"Tokens:   {tokens}")
print(f"Stemmed:  {stemmed}")
print()

# Show how stemming collapses vocabulary
words_group = ["connect", "connected", "connecting", "connection", "connections"]
stems = [stemmer.stem(w) for w in words_group]
print("Vocabulary collapse (good for search/retrieval):")
for w, s in zip(words_group, stems):
    print(f"  {w:<15} -> {s}")
print(f"  All map to: {set(stems)}")

## 6. Code: Optional NLTK Lemmatization

NLTK's `WordNetLemmatizer` uses the WordNet lexical database. We wrap this in a `try/except` since NLTK may not be installed or its data may not be downloaded.

In [ ]:
# ---- NLTK Lemmatizer (guarded import) ----

NLTK_AVAILABLE = False

try:
    import nltk
    from nltk.stem import WordNetLemmatizer, PorterStemmer
    
    # Download required data (suppress output)
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    
    lemmatizer = WordNetLemmatizer()
    porter = PorterStemmer()
    NLTK_AVAILABLE = True
    print("NLTK loaded successfully.")

except ImportError:
    print("NLTK not installed. Skipping NLTK-based examples.")
    print("Install with: pip install nltk")
except Exception as e:
    print(f"NLTK available but data download failed: {e}")
    print("Run: python -m nltk.downloader wordnet omw-1.4")

In [ ]:
# ---- Compare NLTK Porter Stemmer vs WordNet Lemmatizer ----

if NLTK_AVAILABLE:
    test_words = [
        ("running", "v"),  # verb
        ("better", "a"),   # adjective
        ("mice", "n"),     # noun
        ("studies", "n"),  # noun
        ("studying", "v"),  # verb
        ("universities", "n"),  # noun
        ("happiness", "n"),     # noun
        ("was", "v"),      # verb
        ("geese", "n"),    # noun
        ("caring", "v"),   # verb
    ]
    
    print(f"{'Word':<18} {'POS':<6} {'Porter Stem':<18} {'WordNet Lemma':<18}")
    print("-" * 62)
    for word, pos in test_words:
        stemmed = porter.stem(word)
        lemmatized = lemmatizer.lemmatize(word, pos=pos)
        print(f"{word:<18} {pos:<6} {stemmed:<18} {lemmatized:<18}")
    
    print()
    print("Key observations:")
    print("  - Lemmatizer returns valid words; stemmer often does not.")
    print("  - Lemmatizer handles irregular forms (mice->mouse, geese->goose).")
    print("  - Lemmatizer requires POS tag for best results.")
else:
    print("Skipping NLTK comparison (NLTK not available).")
    print("See the manual comparison table in Section 3 above.")

In [ ]:
# ---- Simple dictionary-based lemmatizer (no NLTK needed) ----

class SimpleLemmatizer:
    """A simple dictionary-based lemmatizer for educational purposes.
    
    Uses a hardcoded dictionary of irregular forms plus basic suffix rules.
    """
    
    def __init__(self):
        # Irregular forms lookup
        self.irregulars = {
            # Irregular verbs
            "ran": "run", "was": "be", "were": "be", "been": "be",
            "had": "have", "did": "do", "went": "go", "gone": "go",
            "saw": "see", "seen": "see", "took": "take", "taken": "take",
            "gave": "give", "given": "give", "ate": "eat", "eaten": "eat",
            # Irregular nouns
            "mice": "mouse", "geese": "goose", "feet": "foot",
            "teeth": "tooth", "children": "child", "men": "man",
            "women": "woman", "people": "person", "oxen": "ox",
            # Irregular adjectives/adverbs
            "better": "good", "best": "good",
            "worse": "bad", "worst": "bad",
        }
        # Suffix rules: (suffix, replacement, min_remaining)
        self.suffix_rules = [
            ("ies", "y", 1),      # studies -> study
            ("ves", "f", 1),       # wolves -> wolf
            ("ses", "s", 1),       # houses -> house (approximate)
            ("ing", "", 3),        # running -> runn -> run (needs cleanup)
            ("ed", "", 2),          # walked -> walk
            ("s", "", 2),           # cats -> cat
        ]
    
    def lemmatize(self, word: str) -> str:
        """Lemmatize a single word."""
        word = word.lower().strip()
        
        # Check irregulars first
        if word in self.irregulars:
            return self.irregulars[word]
        
        # Try suffix rules
        for suffix, replacement, min_len in self.suffix_rules:
            if word.endswith(suffix) and len(word) - len(suffix) >= min_len:
                candidate = word[:-len(suffix)] + replacement
                # Clean up doubled consonants (running -> runn -> run)
                if len(candidate) >= 2 and candidate[-1] == candidate[-2]:
                    candidate = candidate[:-1]
                return candidate
        
        return word


simple_lemmatizer = SimpleLemmatizer()

test_words = [
    "running", "cats", "studies", "mice", "better", "was",
    "geese", "walked", "wolves", "children", "went",
]

print("SimpleLemmatizer Results:")
print(f"{'Input':<15} {'Lemma':<15}")
print("-" * 30)
for word in test_words:
    print(f"{word:<15} {simple_lemmatizer.lemmatize(word):<15}")

## 7. Code: Stop Word Removal Effect on Classification

We compare text classification with and without stop word removal to demonstrate when it helps and when it hurts.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# ---- Dataset: Sentiment where negation matters ----
# Deliberately includes negation patterns to show stop word impact

texts = [
    # Positive
    "This movie is really good and enjoyable",
    "Excellent film with great performances",
    "I loved every minute of this movie",
    "The acting was wonderful and the plot was exciting",
    "A truly fantastic and entertaining experience",
    "Great movie that I would recommend to everyone",
    "Beautiful cinematography and amazing story",
    "One of the best films I have ever seen",
    # Negative (many use negation!)
    "This movie is not good at all",           # "not" is a stop word!
    "I did not enjoy this terrible film",       # "not" is a stop word!
    "Not worth watching and not entertaining",  # "not" is a stop word!
    "The acting was not convincing and boring",
    "I would not recommend this to anyone",
    "This is not a good film by any measure",
    "Not the best movie and quite disappointing",
    "The plot was not exciting and very dull",
]
labels = np.array([1]*8 + [0]*8)

print(f"Dataset: {len(texts)} texts ({sum(labels)} positive, {sum(1-labels)} negative)")
print()

# ---- Pipeline 1: WITH stop words (keep all words) ----
pipe_with_stops = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words=None)),  # keep stop words
    ("clf", LogisticRegression(random_state=42, max_iter=1000)),
])

# ---- Pipeline 2: WITHOUT stop words ----
pipe_no_stops = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),  # remove stop words
    ("clf", LogisticRegression(random_state=42, max_iter=1000)),
])

# Evaluate
scores_with = cross_val_score(pipe_with_stops, texts, labels, cv=4, scoring="accuracy")
scores_no = cross_val_score(pipe_no_stops, texts, labels, cv=4, scoring="accuracy")

print("Classification Accuracy:")
print(f"  With stop words:    {scores_with.mean():.3f} (+/- {scores_with.std():.3f})")
print(f"  Without stop words: {scores_no.mean():.3f} (+/- {scores_no.std():.3f})")
print()
print("Analysis:")
print("  When negative samples rely on 'not' for their meaning,")
print("  removing stop words can HURT classification because")
print("  'not good' becomes just 'good' after stop word removal.")

In [ ]:
# ---- Visualize what gets lost ----

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

print("What stop word removal does to negation-heavy sentences:")
print("=" * 60)

negation_examples = [
    "This movie is not good at all",
    "I did not enjoy this terrible film",
    "Not worth watching and not entertaining",
    "No problem with the service",
    "I have no complaints about this product",
]

for text in negation_examples:
    tokens = text.lower().split()
    filtered = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    removed = [t for t in tokens if t in ENGLISH_STOP_WORDS]
    print(f"  Original:  {text}")
    print(f"  Kept:      {' '.join(filtered)}")
    print(f"  Removed:   {removed}")
    print()

In [ ]:
# ---- Compare: stemming + stop words vs lemmatization + keep stops ----

def preprocess_stem_no_stops(text: str) -> str:
    """Stem + remove stop words."""
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    tokens = stemmer.stem_tokens(tokens)
    return " ".join(tokens)

def preprocess_lemma_keep_stops(text: str) -> str:
    """Lemmatize + keep stop words."""
    tokens = text.lower().split()
    tokens = [simple_lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

def preprocess_stem_keep_stops(text: str) -> str:
    """Stem + keep stop words."""
    tokens = text.lower().split()
    tokens = stemmer.stem_tokens(tokens)
    return " ".join(tokens)

# Apply each pipeline
configs = [
    ("Raw text (baseline)", texts),
    ("Stem + remove stops", [preprocess_stem_no_stops(t) for t in texts]),
    ("Stem + keep stops", [preprocess_stem_keep_stops(t) for t in texts]),
    ("Lemma + keep stops", [preprocess_lemma_keep_stops(t) for t in texts]),
]

print("Preprocessing Pipeline Comparison:")
print("=" * 55)
for name, processed_texts in configs:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(random_state=42, max_iter=1000)),
    ])
    scores = cross_val_score(pipe, processed_texts, labels, cv=4, scoring="accuracy")
    print(f"  {name:<25s}  Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

print()
print("Takeaway: The best pipeline depends on your data and task.")
print("When negation matters, keeping stop words tends to help.")

## 8. Common Mistakes & Debugging Tips

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Using stemming for chatbot/QA | Output has non-words like `"univers"` | Use lemmatization instead |
| Not providing POS to lemmatizer | `"better"` stays `"better"` instead of `"good"` | Pass POS tag: `lemmatize("better", pos="a")` |
| Removing stop words for deep learning | Loses negation: `"not good"` -> `"good"` | Keep stop words for contextual models |
| Removing stop words for QA | Loses question words: `"who"`, `"what"` | Keep them or use custom stop list |
| Over-stemming | Unrelated words merge: `"universe"` = `"university"` | Use lemmatization or lighter stemmer |
| Under-stemming | Related words stay separate: `"alumnus"` != `"alumni"` | Use lemmatization with lookup tables |
| Applying stemming/lemma AFTER subword tokenization | Corrupts subword tokens | Always stem/lemmatize before tokenization |
| Using a fixed stop word list across languages | Misses language-specific patterns | Use language-appropriate stop word lists |

**Debugging tip:** Always inspect a sample of your preprocessed text before training. Print 10-20 examples to catch obvious issues like lost negation or mangled words.

## 9. Exercises

### Exercise 1: Compare Preprocessing Pipelines

Given the dataset below, create and evaluate four preprocessing pipelines:
1. Raw text (no preprocessing)
2. Stemming only
3. Stemming + stop word removal
4. Lemmatization only

Report the TF-IDF classification accuracy for each.

In [ ]:
exercise_texts = [
    "The researchers are studying language models",
    "These studies of languages were groundbreaking",
    "Running experiments on computational models was exciting",
    "The runner ran multiple experiments successfully",
    "Natural language processing is not easy to learn",
    "Learning natural languages does not come easily",
    "The cats and dogs are not fighting anymore",
    "Those mice were not running from the cats",
    "Bad weather was not expected for the weekend",
    "The worst storms did not hit the coastal areas",
    "Universities are connecting students with better opportunities",
    "The university connections helped improve student happiness",
]
exercise_labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1])

# TODO: Create 4 preprocessing functions and evaluate each
# Your code here

### Exercise 2: Custom Stop Word List

Create a custom stop word list that **keeps negation words** (`not`, `no`, `nor`, `neither`, `never`) but removes other common stop words. Compare its classification performance against full removal and no removal.

In [ ]:
# TODO: Build a custom stop word list and compare
# Hint: Start with sklearn's ENGLISH_STOP_WORDS and remove negation words
# Your code here

## 10. Exercise Solutions

In [ ]:
# ---- Exercise 1 Solution ----

def pipe_raw(text):
    return text.lower()

def pipe_stem(text):
    tokens = text.lower().split()
    return " ".join(stemmer.stem_tokens(tokens))

def pipe_stem_nostop(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in ENGLISH_STOP_WORDS]
    return " ".join(stemmer.stem_tokens(tokens))

def pipe_lemma(text):
    tokens = text.lower().split()
    return " ".join([simple_lemmatizer.lemmatize(t) for t in tokens])

pipelines = [
    ("1. Raw", [pipe_raw(t) for t in exercise_texts]),
    ("2. Stem only", [pipe_stem(t) for t in exercise_texts]),
    ("3. Stem + no stops", [pipe_stem_nostop(t) for t in exercise_texts]),
    ("4. Lemma only", [pipe_lemma(t) for t in exercise_texts]),
]

print("Exercise 1: Pipeline Comparison")
print("=" * 55)
for name, proc_texts in pipelines:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("clf", LogisticRegression(random_state=42, max_iter=1000)),
    ])
    scores = cross_val_score(pipe, proc_texts, exercise_labels, cv=3, scoring="accuracy")
    print(f"  {name:<22s}  Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# Show sample preprocessed text
print("\nSample preprocessing (first text):")
for name, proc_texts in pipelines:
    print(f"  {name}: {proc_texts[0]}")

In [ ]:
# ---- Exercise 2 Solution ----

# Keep negation words
NEGATION_WORDS = {"not", "no", "nor", "neither", "never", "nobody", "nothing", "nowhere"}
CUSTOM_STOP_WORDS = sorted(ENGLISH_STOP_WORDS - NEGATION_WORDS)

print(f"Standard stop words: {len(ENGLISH_STOP_WORDS)}")
print(f"Custom stop words (negation kept): {len(CUSTOM_STOP_WORDS)}")
print(f"Negation words kept: {NEGATION_WORDS & ENGLISH_STOP_WORDS}")
print()

# Compare three approaches
configs = [
    ("No removal", TfidfVectorizer(stop_words=None)),
    ("Full removal", TfidfVectorizer(stop_words="english")),
    ("Custom (keep negation)", TfidfVectorizer(stop_words=CUSTOM_STOP_WORDS)),
]

print("Exercise 2: Custom Stop Word List")
print("=" * 55)
for name, vectorizer in configs:
    pipe = Pipeline([
        ("tfidf", vectorizer),
        ("clf", LogisticRegression(random_state=42, max_iter=1000)),
    ])
    scores = cross_val_score(pipe, texts, labels, cv=4, scoring="accuracy")
    print(f"  {name:<25s}  Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

print()
print("The custom list often provides a good balance:")
print("  - Removes noise words (the, is, a)")
print("  - Keeps negation (not, no, never) for sentiment")